# LLaMA: Open and Efficient Foundation Language Models

---

## Paper Reference

**Title:** LLaMA: Open and Efficient Foundation Language Models  
**Authors:** Hugo Touvron, Thibaut Lavril, Gautier Izacard, Xavier Martinet, Marie-Anne Lachaux, Timothée Lacroix, Baptiste Rozière, Naman Goyal, Eric Hambro, Faisal Azhar, Aurelien Rodriguez, Armand Joulin, Edouard Grave, Guillaume Lample  
**Affiliation:** Meta AI  
**Published:** February 2023  
**arXiv:** [2302.13971](https://arxiv.org/abs/2302.13971)  

---

## What This Notebook Covers

This notebook provides a **complete, from-scratch implementation** of the LLaMA architecture with detailed explanations tied to every section and equation in the paper. We will:

1. Understand the **core thesis** — why training smaller models on more data beats larger models at inference
2. Implement every architectural component: **RMSNorm**, **SwiGLU**, **RoPE**, **Multi-Head Attention**
3. Assemble the full **LLaMA Transformer** and verify parameter counts match Table 2
4. Reproduce training details: **cosine LR schedule**, **data mix**, **efficiency tricks**
5. Visualize **benchmark results** and **training curves** from the paper

## Key Contribution

> *"We show that it is possible to train state-of-the-art models using **publicly available datasets exclusively**, without resorting to proprietary and inaccessible datasets."* — §1

LLaMA demonstrated that a **7B parameter model** trained on **1 trillion tokens** of public data could match or exceed GPT-3 (175B) on most benchmarks, fundamentally challenging the notion that model size is the primary scaling lever.

## 2. Setup & Imports

### WHAT
We import the core libraries needed to implement LLaMA from scratch.

### WHY
- **PyTorch** (`torch`, `torch.nn`, `torch.nn.functional`): The deep learning framework for tensor operations, neural network modules, and activation functions.
- **matplotlib**: For visualizing activations, learning rate schedules, training curves, and benchmark comparisons.
- **numpy**: For numerical computations outside the autograd graph (e.g., generating synthetic data).
- **math**: For mathematical constants ($\pi$, square roots) used in positional embeddings.
- **dataclass**: For clean, typed configuration objects matching the paper's Table 2.
- **typing**: For type annotations that make the code self-documenting.

### HOW
Standard Python imports — no external custom packages required. The entire notebook is self-contained.

### WHERE
These are foundational tools used throughout every section of the paper's implementation.

In [ ]:
# ============================================================
# Core imports for the LLaMA implementation
# ============================================================

import torch                          # Core tensor library
import torch.nn as nn                 # Neural network modules (Linear, Embedding, etc.)
import torch.nn.functional as F       # Functional API (softmax, cross_entropy, etc.)
import matplotlib.pyplot as plt       # Plotting and visualization
import matplotlib.patches as mpatches # For custom legend patches in plots
import numpy as np                    # Numerical computing
import math                           # Mathematical constants and functions
from typing import Optional, Tuple    # Type hints for function signatures
from dataclasses import dataclass     # Clean configuration classes

# ============================================================
# Device selection: use GPU if available, else CPU
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 3. The Core Thesis — Scaling Laws Revisited (§1)

### WHAT
The LLaMA paper challenges the prevailing **Chinchilla scaling laws** (Hoffmann et al., 2022) by arguing that the optimal model size depends on your **inference budget**, not just your training compute budget.

### WHY
The Chinchilla scaling law says: for a given compute budget $C$, there exist optimal model size $N$ and dataset size $D$ such that:

$$C \approx 6ND$$

This means you should **scale model parameters and training tokens equally**. For example, a 70B model should be trained on ~1.4T tokens.

But LLaMA's insight is different:

> *"The performance of a 7B model continues to improve even after 1T tokens."* — §1

At **inference time**, a smaller model is cheaper to serve. So the right question is:

> **Given a target inference budget (model size), what is the best performance achievable by training on more data?**

### HOW
LLaMA trains models that are **"over-trained" by Chinchilla standards**:

| Model | Parameters | Training Tokens | Chinchilla-Optimal Tokens |
|-------|-----------|----------------|-------------------------|
| LLaMA-7B | 6.7B | 1.0T | ~140B |
| LLaMA-13B | 13.0B | 1.0T | ~260B |
| LLaMA-33B | 32.5B | 1.4T | ~650B |
| LLaMA-65B | 65.2B | 1.4T | ~1.3T |

The 7B model sees **7× more tokens** than Chinchilla would recommend!

### WHERE
This is the central argument of §1 (Introduction) and validated throughout §3 (Main Results) and §4 (Instruction Finetuning).

### Key Results
- **LLaMA-13B outperforms GPT-3 (175B)** on most benchmarks
- **LLaMA-65B is competitive with Chinchilla-70B and PaLM-540B**
- All trained on **publicly available data only**

## 4. RMSNorm — Root Mean Square Normalization (§2.2)

### WHAT
RMSNorm (Zhang & Sennrich, 2019) is a simplified version of Layer Normalization that **only rescales** activations without re-centering them (no mean subtraction).

### WHY
Standard LayerNorm computes:

$$\text{LayerNorm}(\mathbf{a}) = \frac{\mathbf{a} - \mu}{\sigma} \odot \mathbf{g} + \mathbf{b}$$

where $\mu = \frac{1}{n}\sum_i a_i$ and $\sigma = \sqrt{\frac{1}{n}\sum_i (a_i - \mu)^2}$.

RMSNorm **drops the mean subtraction** and bias, computing only:

$$\bar{a}_i = \frac{a_i}{\text{RMS}(\mathbf{a})} \cdot g_i$$

where:

$$\text{RMS}(\mathbf{a}) = \sqrt{\frac{1}{n} \sum_{i=1}^{n} a_i^2}$$

This is **computationally cheaper** (one fewer reduction operation) while achieving **comparable training stability**. The key insight from Zhang & Sennrich (2019) is that the re-centering in LayerNorm is unnecessary — the re-scaling does all the heavy lifting.

### HOW
1. Compute the mean of squared activations along the last dimension
2. Take the reciprocal square root (with epsilon for numerical stability)
3. Multiply element-wise by learnable gain parameters $g_i$

### WHERE
From §2.2: *"We normalize the input of each transformer sub-layer... We use the RMSNorm normalizing function (Zhang and Sennrich, 2019)."*

RMSNorm is applied as **pre-normalization** (before each sub-layer), following the GPT-3 / PaLM pattern rather than the original Transformer's post-normalization.

In [ ]:
# ============================================================
# RMSNorm Implementation (§2.2)
# ============================================================

class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization.
    
    Unlike LayerNorm, RMSNorm does NOT subtract the mean.
    It only rescales by the RMS of the activations.
    
    Equation:
        ā_i = (a_i / RMS(a)) * g_i
        RMS(a) = sqrt(1/n * Σ a_i²)
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps                              # Small constant for numerical stability
        self.weight = nn.Parameter(torch.ones(dim)) # Learnable gain parameter g_i
    
    def _norm(self, x: torch.Tensor) -> torch.Tensor:
        """
        Compute x / RMS(x).
        
        We use rsqrt (reciprocal square root) for efficiency:
            1/sqrt(mean(x²) + eps) is computed in one fused op.
        """
        # x.pow(2):           Square each element → a_i²
        # .mean(-1, ...):     Average over last dim → (1/n)Σa_i²
        # .rsqrt():           Reciprocal sqrt → 1/RMS(a)
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass: normalize then scale by learnable weights.
        
        We cast to float32 for the norm computation to avoid
        numerical issues with float16/bfloat16, then cast back.
        """
        # Cast to float32 for stable norm computation
        output = self._norm(x.float()).type_as(x)
        # Element-wise multiply by learnable gain: ā_i * g_i
        return output * self.weight


# ============================================================
# Demonstration: Compare RMSNorm vs LayerNorm
# ============================================================
print("=" * 60)
print("RMSNorm vs LayerNorm Comparison")
print("=" * 60)

dim = 8
x = torch.randn(2, 4, dim)  # (batch=2, seq_len=4, dim=8)

# Our RMSNorm
rmsnorm = RMSNorm(dim)
rms_out = rmsnorm(x)

# PyTorch's LayerNorm (with elementwise_affine=True by default)
layernorm = nn.LayerNorm(dim)
ln_out = layernorm(x)

print(f"\nInput shape: {x.shape}")
print(f"Input sample (first token): {x[0, 0, :].tolist()}")
print(f"\nRMSNorm output (first token): {rms_out[0, 0, :].detach().tolist()}")
print(f"LayerNorm output (first token): {ln_out[0, 0, :].detach().tolist()}")

# Verify: RMSNorm output should have unit RMS (before gain)
normed = rmsnorm._norm(x.float())
rms_of_output = torch.sqrt(normed.pow(2).mean(-1))  # Should be ~1.0
print(f"\nRMS of normalized output (should be ~1.0): {rms_of_output[0, 0].item():.6f}")

# Key difference: RMSNorm does NOT center the data
print(f"\nMean of RMSNorm output: {rms_out[0, 0].mean().item():.4f} (NOT necessarily 0)")
print(f"Mean of LayerNorm output: {ln_out[0, 0].mean().item():.4f} (close to 0)")

# Computational savings: RMSNorm skips mean computation
print(f"\n--- Computational Comparison ---")
print(f"LayerNorm ops: mean reduction + variance reduction + normalize + affine")
print(f"RMSNorm ops:   mean-of-squares reduction + normalize + scale (no bias)")
print(f"Savings: 1 fewer reduction (mean), 1 fewer addition (bias term)")
print(f"RMSNorm parameters: {sum(p.numel() for p in rmsnorm.parameters())} (gain only)")
print(f"LayerNorm parameters: {sum(p.numel() for p in layernorm.parameters())} (gain + bias)")

## 5. SwiGLU Activation Function (§2.2)

### WHAT
SwiGLU is a **gated linear unit** variant that uses the **SiLU (Swish)** activation as its gating function. It replaces the standard ReLU-based feed-forward network in each Transformer layer.

### WHY
From §2.2: *"We replace the ReLU non-linearity by the SwiGLU activation function, introduced by Shazeer (2020), to improve the performance."*

Shazeer (2020) showed that gated activations consistently outperform simple point-wise activations (ReLU, GELU) in Transformer language models. The intuition is that the **gating mechanism** allows the network to learn which features to pass through, providing a richer representational capacity.

### HOW

**Standard FFN** (original Transformer):
$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

This uses two weight matrices: $W_1 \in \mathbb{R}^{d \times 4d}$ and $W_2 \in \mathbb{R}^{4d \times d}$.

**SwiGLU FFN** (LLaMA):
$$\text{FFN}_{\text{SwiGLU}}(x, W, V, W_2) = (\text{SiLU}(xW) \otimes xV) W_2$$

where:
- $\text{SiLU}(x) = x \cdot \sigma(x)$ is the Sigmoid Linear Unit (also called Swish-1)
- $\sigma(x) = \frac{1}{1 + e^{-x}}$ is the sigmoid function
- $\otimes$ denotes element-wise multiplication
- $W, V \in \mathbb{R}^{d \times d_{\text{ff}}}$ are the gate and value projections
- $W_2 \in \mathbb{R}^{d_{\text{ff}} \times d}$ is the down-projection

**The $\frac{2}{3} \cdot 4d$ trick:** Since SwiGLU uses **three** weight matrices instead of two, the hidden dimension is adjusted from $4d$ to $\frac{2}{3} \cdot 4d = \frac{8d}{3}$ to keep the total parameter count roughly the same. This is then rounded to the nearest `multiple_of` (256) for hardware efficiency.

### WHERE
§2.2: Every Transformer layer's feed-forward sub-block uses SwiGLU instead of a standard FFN.

In [ ]:
# ============================================================
# SwiGLU Feed-Forward Network Implementation (§2.2)
# ============================================================

class FeedForward(nn.Module):
    """
    SwiGLU Feed-Forward Network as used in LLaMA.
    
    FFN_SwiGLU(x, W, V, W2) = (SiLU(xW) ⊗ xV) W2
    
    Three linear projections:
        w1 (gate):  x → hidden_dim  (the "W" in the equation)
        w3 (value): x → hidden_dim  (the "V" in the equation)  
        w2 (down):  hidden_dim → x  (the "W2" in the equation)
    """
    def __init__(self, dim: int, hidden_dim: int, multiple_of: int = 256):
        super().__init__()
        
        # Apply the 2/3 * 4d trick to keep param count comparable to standard FFN
        # Standard FFN: 2 * dim * 4*dim = 8 * dim² parameters
        # SwiGLU FFN:   3 * dim * hidden_dim parameters
        # Setting hidden_dim = (2/3) * 4 * dim gives 3 * dim * (8/3)*dim = 8*dim²
        hidden_dim = int(2 * hidden_dim / 3)
        
        # Round up to nearest multiple_of for hardware efficiency (tensor core alignment)
        hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of)
        
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)  # Gate projection (W)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)   # Down projection (W2)
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)  # Value projection (V)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        SwiGLU forward pass:
            1. Compute gate: SiLU(x @ W1)
            2. Compute value: x @ W3
            3. Element-wise gate the value: gate * value
            4. Project down: result @ W2
        """
        # F.silu(x) = x * sigmoid(x), the SiLU/Swish activation
        gate = F.silu(self.w1(x))    # SiLU(xW) — gating signal
        value = self.w3(x)            # xV — value signal
        return self.w2(gate * value)  # (SiLU(xW) ⊗ xV) W2


# ============================================================
# Standard FFN for comparison
# ============================================================
class StandardFFN(nn.Module):
    """Standard Transformer FFN: ReLU(xW1)W2"""
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.relu(self.w1(x)))


# ============================================================
# Compare parameter counts
# ============================================================
dim = 512
hidden_dim = 4 * dim  # Standard expansion factor

swiglu_ffn = FeedForward(dim, hidden_dim)
standard_ffn = StandardFFN(dim, hidden_dim)

swiglu_params = sum(p.numel() for p in swiglu_ffn.parameters())
standard_params = sum(p.numel() for p in standard_ffn.parameters())

print("=" * 60)
print("SwiGLU vs Standard FFN Comparison")
print("=" * 60)
print(f"\nModel dimension: {dim}")
print(f"Standard FFN hidden dim: {hidden_dim}")
print(f"SwiGLU FFN hidden dim (after 2/3 adjustment): {swiglu_ffn.w1.out_features}")
print(f"\nStandard FFN parameters: {standard_params:,} ({standard_params/1e6:.2f}M)")
print(f"SwiGLU FFN parameters:   {swiglu_params:,} ({swiglu_params/1e6:.2f}M)")
print(f"Ratio (SwiGLU/Standard): {swiglu_params/standard_params:.3f}")

# Test forward pass
x = torch.randn(2, 4, dim)  # (batch=2, seq_len=4, dim=512)
out = swiglu_ffn(x)
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Output matches input dim: {x.shape == out.shape}")

In [ ]:
# ============================================================
# Visualization: SiLU vs ReLU vs GELU
# ============================================================
# This helps understand WHY SiLU is preferred:
# - Smooth everywhere (unlike ReLU's hard corner at 0)
# - Non-monotonic: slightly negative for small negative inputs
#   → allows gradient flow for slightly negative values
# - Self-gated: x * sigmoid(x) means the input gates itself

x_plot = torch.linspace(-5, 5, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Activation functions
axes[0].plot(x_plot.numpy(), F.relu(x_plot).numpy(), label='ReLU', linewidth=2, color='#e74c3c')
axes[0].plot(x_plot.numpy(), F.gelu(x_plot).numpy(), label='GELU', linewidth=2, color='#2ecc71', linestyle='--')
axes[0].plot(x_plot.numpy(), F.silu(x_plot).numpy(), label='SiLU (Swish)', linewidth=2, color='#3498db')
axes[0].axhline(y=0, color='gray', linewidth=0.5, linestyle='-')
axes[0].axvline(x=0, color='gray', linewidth=0.5, linestyle='-')
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('f(x)', fontsize=12)
axes[0].set_title('Activation Functions: SiLU vs ReLU vs GELU', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-1, 5)

# Right: Derivatives (gradient flow)
x_grad = x_plot.clone().requires_grad_(True)

# Compute gradients for each activation
for act_fn, name, color, ls in [
    (F.relu, 'ReLU', '#e74c3c', '-'),
    (F.gelu, 'GELU', '#2ecc71', '--'),
    (F.silu, 'SiLU (Swish)', '#3498db', '-')
]:
    x_g = x_plot.clone().requires_grad_(True)
    y = act_fn(x_g).sum()
    y.backward()
    axes[1].plot(x_plot.numpy(), x_g.grad.numpy(), label=name, linewidth=2, color=color, linestyle=ls)

axes[1].axhline(y=0, color='gray', linewidth=0.5)
axes[1].axvline(x=0, color='gray', linewidth=0.5)
axes[1].set_xlabel('x', fontsize=12)
axes[1].set_ylabel("f'(x)", fontsize=12)
axes[1].set_title('Derivatives: Gradient Flow Properties', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print("• ReLU: Hard cutoff at 0, zero gradient for x < 0 (dying ReLU problem)")
print("• GELU: Smooth approximation of ReLU, small gradient for negative inputs")
print("• SiLU: Smoothly self-gated, non-zero gradient for slightly negative inputs")
print("  → SiLU's non-monotonicity allows it to 'softly reject' small negative values")

## 6. Rotary Position Embeddings (RoPE) — §2.2

### WHAT
**Rotary Position Embeddings** (RoPE, Su et al. 2021) encode position information by **rotating** query and key vectors in the complex plane. This is applied at **every attention layer** rather than adding positional embeddings only once at the input.

### WHY
From §2.2: *"We remove the absolute positional embeddings, and instead, add rotary positional embeddings (RoPE), introduced by Su et al. (2021), at each layer of the network."*

RoPE has several advantages over absolute positional embeddings:
1. **Relative position awareness**: The dot product $\mathbf{q}_m \cdot \mathbf{k}_n$ depends only on the relative position $m - n$, not the absolute positions
2. **Decaying with distance**: Tokens far apart naturally attend less to each other
3. **Extrapolation**: Better generalization to sequence lengths unseen during training

### HOW
For a $d$-dimensional vector, RoPE groups dimensions into $d/2$ pairs and applies a 2D rotation with position-dependent angle to each pair.

The rotation angles are:
$$\theta_i = 10000^{-2i/d}, \quad i = 0, 1, \ldots, d/2 - 1$$

For position $m$, the rotation applied to the $i$-th pair $(x_{2i}, x_{2i+1})$ is:

$$\begin{pmatrix} x'_{2i} \\ x'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$

Using complex number notation (as in the LLaMA implementation), this becomes:
$$(x_{2i} + jx_{2i+1}) \cdot e^{jm\theta_i} = (x_{2i} + jx_{2i+1})(\cos(m\theta_i) + j\sin(m\theta_i))$$

### WHERE
§2.2: Applied to queries and keys at **each** attention layer. Not applied to values.

In [ ]:
# ============================================================
# Rotary Position Embeddings (RoPE) Implementation (§2.2)
# ============================================================

def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0) -> torch.Tensor:
    """
    Precompute the complex exponentials e^{j * m * theta_i} for RoPE.
    
    Args:
        dim:   Head dimension (each head's query/key size)
        end:   Maximum sequence length
        theta: Base for the geometric frequency series (10000 in the paper)
    
    Returns:
        Complex tensor of shape (end, dim//2) containing e^{j*m*theta_i}
    """
    # Compute the frequency for each dimension pair:
    # theta_i = 10000^(-2i/d) for i = 0, 1, ..., d/2 - 1
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    
    # Create position indices: m = 0, 1, ..., end-1
    t = torch.arange(end, device=freqs.device)
    
    # Outer product: (end, dim//2) matrix of m * theta_i
    freqs = torch.outer(t, freqs).float()
    
    # Convert to complex exponentials: e^{j * m * theta_i}
    # polar(r, theta) = r * (cos(theta) + j*sin(theta))
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis


def reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """
    Reshape frequency tensor for broadcasting with query/key tensors.
    
    freqs_cis shape: (seq_len, head_dim//2)
    x shape:         (batch, seq_len, n_heads, head_dim//2)
    
    We need freqs_cis to be (1, seq_len, 1, head_dim//2) for broadcasting.
    """
    ndim = x.ndim
    assert ndim >= 2
    # Build shape: 1 for all dims except seq_len (dim 1) and head_dim (last dim)
    shape = [1 if i != 1 and i != ndim - 1 else d for i, d in enumerate(x.shape)]
    return freqs_cis.view(*shape)


def apply_rotary_emb(
    xq: torch.Tensor,          # Query tensor: (batch, seq_len, n_heads, head_dim)
    xk: torch.Tensor,          # Key tensor:   (batch, seq_len, n_heads, head_dim)
    freqs_cis: torch.Tensor    # Precomputed frequencies: (seq_len, head_dim//2)
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Apply rotary embeddings to query and key tensors using complex multiplication.
    
    The trick: view pairs of real dimensions as complex numbers,
    multiply by e^{j*m*theta}, then convert back to real.
    
    Steps:
        1. Reshape (batch, seq, heads, dim) → (batch, seq, heads, dim//2, 2)
        2. View as complex: (batch, seq, heads, dim//2) complex
        3. Multiply by e^{j*m*theta_i} (this IS the rotation)
        4. View as real and flatten back to (batch, seq, heads, dim)
    """
    # Step 1-2: Pair up dimensions and interpret as complex numbers
    # e.g., (x0, x1) → x0 + j*x1
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    
    # Reshape frequencies for broadcasting across batch and head dimensions
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
    
    # Step 3: Apply rotation via complex multiplication
    # (a + jb)(cos θ + j sin θ) = (a cos θ - b sin θ) + j(a sin θ + b cos θ)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)  # Back to real
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    
    return xq_out.type_as(xq), xk_out.type_as(xk)


# ============================================================
# Demonstration: Verify RoPE properties
# ============================================================
print("=" * 60)
print("RoPE: Rotary Position Embedding Verification")
print("=" * 60)

head_dim = 16
max_seq = 64

# Precompute frequencies
freqs = precompute_freqs_cis(head_dim, max_seq)
print(f"\nFrequency tensor shape: {freqs.shape}")
print(f"  → (max_seq_len={max_seq}, head_dim//2={head_dim//2})")

# Create synthetic q, k at different positions
batch, seq_len, n_heads = 1, 8, 2
q = torch.randn(batch, seq_len, n_heads, head_dim)
k = torch.randn(batch, seq_len, n_heads, head_dim)

# Apply RoPE
q_rot, k_rot = apply_rotary_emb(q, k, freqs[:seq_len])
print(f"\nQuery shape before RoPE: {q.shape}")
print(f"Query shape after RoPE:  {q_rot.shape}")

# ============================================================
# KEY PROPERTY: q·k depends only on relative position (m-n)
# ============================================================
# Take a single head's q at position 2, k at position 5: relative distance = 3
# Compare with q at position 4, k at position 7: also relative distance = 3

# Use the SAME content vectors at different positions
q_content = torch.randn(1, 1, 1, head_dim)  # Single content vector
k_content = torch.randn(1, 1, 1, head_dim)  # Single content vector

# Place at positions (2, 5) — relative distance 3
q1_rot, _ = apply_rotary_emb(q_content, q_content, freqs[2:3])
_, k1_rot = apply_rotary_emb(k_content, k_content, freqs[5:6])
dot1 = (q1_rot * k1_rot).sum().item()

# Place at positions (4, 7) — also relative distance 3
q2_rot, _ = apply_rotary_emb(q_content, q_content, freqs[4:5])
_, k2_rot = apply_rotary_emb(k_content, k_content, freqs[7:8])
dot2 = (q2_rot * k2_rot).sum().item()

print(f"\n--- Relative Position Property ---")
print(f"q·k at positions (2, 5), relative dist = 3: {dot1:.6f}")
print(f"q·k at positions (4, 7), relative dist = 3: {dot2:.6f}")
print(f"Match (should be True): {abs(dot1 - dot2) < 1e-4}")
print(f"  → Same content at same relative distance gives same dot product!")

## 7. Multi-Head Attention with RoPE (§2.2)

### WHAT
The multi-head self-attention mechanism is the core component of the Transformer. In LLaMA, it integrates **RoPE** (applied to queries and keys) and uses a **causal mask** for autoregressive language modeling.

### WHY
Attention allows each token to gather information from all previous tokens in the sequence. The multi-head variant allows the model to attend to different types of relationships (syntactic, semantic, positional) in parallel through different heads.

The standard attention equation is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

With RoPE, queries and keys are rotated before computing the attention scores:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{R_\Theta Q (R_\Theta K)^T}{\sqrt{d_k}}\right) V$$

where $R_\Theta$ denotes the rotary position encoding transformation.

### HOW
1. Project input to Q, K, V using three separate linear layers (no bias, per LLaMA)
2. Reshape to separate heads
3. Apply RoPE to Q and K (not V!)
4. Compute scaled dot-product attention with causal mask
5. Concatenate heads and project output

### WHERE
§2.2: This is the attention mechanism used in every Transformer layer. The paper notes no specific attention modifications beyond RoPE — the architecture is straightforward multi-head attention.

In [ ]:
# ============================================================
# Multi-Head Attention with RoPE (§2.2)
# ============================================================

class Attention(nn.Module):
    """
    Multi-Head Self-Attention with Rotary Position Embeddings.
    
    Key design choices from LLaMA:
        - No bias in linear projections
        - RoPE applied to Q and K only (not V)
        - Causal masking for autoregressive generation
        - No dropout (LLaMA doesn't use attention dropout)
    """
    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads  # d_k = d_v = dim / n_heads
        
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        
        # Q, K, V projections — all without bias (LLaMA convention)
        self.wq = nn.Linear(dim, dim, bias=False)
        self.wk = nn.Linear(dim, dim, bias=False)
        self.wv = nn.Linear(dim, dim, bias=False)
        self.wo = nn.Linear(dim, dim, bias=False)  # Output projection
    
    def forward(
        self,
        x: torch.Tensor,              # (batch, seq_len, dim)
        freqs_cis: torch.Tensor,       # (seq_len, head_dim//2) complex
        mask: Optional[torch.Tensor] = None  # Causal mask
    ) -> torch.Tensor:
        batch, seq_len, _ = x.shape
        
        # Step 1: Linear projections
        # Shape: (batch, seq_len, dim) → (batch, seq_len, dim)
        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)
        
        # Step 2: Reshape to separate heads
        # (batch, seq_len, dim) → (batch, seq_len, n_heads, head_dim)
        q = q.view(batch, seq_len, self.n_heads, self.head_dim)
        k = k.view(batch, seq_len, self.n_heads, self.head_dim)
        v = v.view(batch, seq_len, self.n_heads, self.head_dim)
        
        # Step 3: Apply RoPE to Q and K only
        # This encodes positional information directly into the attention scores
        q, k = apply_rotary_emb(q, k, freqs_cis)
        
        # Step 4: Transpose for attention computation
        # (batch, seq_len, n_heads, head_dim) → (batch, n_heads, seq_len, head_dim)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        # Step 5: Scaled dot-product attention
        # scores = Q K^T / sqrt(d_k)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Apply causal mask: prevent attending to future tokens
        if mask is not None:
            scores = scores + mask  # mask has -inf for future positions
        
        # Softmax over the key dimension
        attn_weights = F.softmax(scores.float(), dim=-1).type_as(q)
        
        # Weighted sum of values
        # (batch, n_heads, seq_len, head_dim)
        output = torch.matmul(attn_weights, v)
        
        # Step 6: Concatenate heads and project
        # (batch, n_heads, seq_len, head_dim) → (batch, seq_len, dim)
        output = output.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        
        return self.wo(output)


# ============================================================
# Demonstration: Attention with RoPE
# ============================================================
print("=" * 60)
print("Multi-Head Attention with RoPE")
print("=" * 60)

dim = 64
n_heads = 4
seq_len = 8
batch = 2

attn = Attention(dim, n_heads)
x = torch.randn(batch, seq_len, dim)

# Precompute RoPE frequencies
freqs_cis = precompute_freqs_cis(dim // n_heads, seq_len)

# Create causal mask: upper triangular with -inf
mask = torch.full((seq_len, seq_len), float("-inf"))
mask = torch.triu(mask, diagonal=1)  # Zero on/below diagonal, -inf above
mask = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len) for broadcasting

out = attn(x, freqs_cis, mask)

print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Number of heads: {n_heads}")
print(f"Head dimension: {dim // n_heads}")
print(f"\nAttention parameters: {sum(p.numel() for p in attn.parameters()):,}")
print(f"  wq: {attn.wq.weight.shape}")
print(f"  wk: {attn.wk.weight.shape}")
print(f"  wv: {attn.wv.weight.shape}")
print(f"  wo: {attn.wo.weight.shape}")

# Visualize the causal mask
print(f"\nCausal mask (0 = attend, -inf = block):")
mask_vis = mask[0, 0].clone()
mask_vis[mask_vis == float('-inf')] = -1  # For display
mask_vis[mask_vis == 0] = 1
for i in range(seq_len):
    row = ['  ✓' if mask_vis[i, j] == 1 else '  ✗' for j in range(seq_len)]
    print(f"  pos {i}: {''.join(row)}")

## 8. The LLaMA Transformer Block (§2)

### WHAT
A single Transformer block in LLaMA combines:
1. **Pre-normalization** with RMSNorm (before attention)
2. Multi-head self-attention with RoPE
3. Residual connection
4. **Pre-normalization** with RMSNorm (before FFN)
5. SwiGLU feed-forward network
6. Residual connection

### WHY
**Pre-normalization** (normalizing the input to each sub-layer rather than the output) was introduced by Xiong et al. (2020) and adopted by GPT-2/3 and PaLM. It provides **more stable training** because:
- Gradients flow through the residual path without being scaled by normalization
- The network can be seen as a sum of terms of increasing complexity

### HOW
The data flow through one block is:

```
x → [RMSNorm] → [Attention + RoPE] → (+x) → [RMSNorm] → [SwiGLU FFN] → (+x) → output
     ↑ pre-norm     ↑ sub-layer 1     ↑ residual  ↑ pre-norm    ↑ sub-layer 2    ↑ residual
```

Mathematically:
$$h = x + \text{Attention}(\text{RMSNorm}(x))$$
$$\text{out} = h + \text{FFN}_{\text{SwiGLU}}(\text{RMSNorm}(h))$$

### WHERE
§2.2: *"Following [GPT-3], we use pre-normalization... We use the RMSNorm normalizing function."*

Each of the $N$ layers in Table 2 is one such block (32 layers for 7B, up to 80 for 65B).

In [ ]:
# ============================================================
# LLaMA Transformer Block (§2)
# ============================================================

class TransformerBlock(nn.Module):
    """
    One Transformer layer in LLaMA.
    
    Architecture:
        x → RMSNorm → Attention(+RoPE) → +residual
          → RMSNorm → SwiGLU FFN       → +residual → output
    
    This is the "pre-norm" variant (GPT-2/3 style),
    NOT the original Transformer's "post-norm".
    """
    def __init__(self, dim: int, n_heads: int, hidden_dim: int,
                 multiple_of: int = 256, norm_eps: float = 1e-6):
        super().__init__()
        
        # Sub-layer 1: Multi-head attention with RoPE
        self.attention = Attention(dim, n_heads)
        
        # Sub-layer 2: SwiGLU feed-forward
        self.feed_forward = FeedForward(dim, hidden_dim, multiple_of)
        
        # Pre-normalization layers (one before each sub-layer)
        self.attention_norm = RMSNorm(dim, eps=norm_eps)  # Before attention
        self.ffn_norm = RMSNorm(dim, eps=norm_eps)        # Before FFN
    
    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        # Sub-layer 1: Pre-norm → Attention → Residual
        # h = x + Attention(RMSNorm(x))
        h = x + self.attention(self.attention_norm(x), freqs_cis, mask)
        
        # Sub-layer 2: Pre-norm → FFN → Residual
        # out = h + FFN(RMSNorm(h))
        out = h + self.feed_forward(self.ffn_norm(h))
        
        return out


# ============================================================
# Demonstration: Data flow through a single block
# ============================================================
print("=" * 60)
print("LLaMA Transformer Block")
print("=" * 60)

dim = 128
n_heads = 4
hidden_dim = 4 * dim
seq_len = 8
batch = 2

block = TransformerBlock(dim, n_heads, hidden_dim)
x = torch.randn(batch, seq_len, dim)
freqs_cis = precompute_freqs_cis(dim // n_heads, seq_len)
mask = torch.full((seq_len, seq_len), float("-inf"))
mask = torch.triu(mask, diagonal=1).unsqueeze(0).unsqueeze(0)

out = block(x, freqs_cis, mask)

print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"\n--- Data Flow Diagram ---")
print(f"")
print(f"  Input x: {list(x.shape)}")
print(f"    │")
print(f"    ├──→ RMSNorm (attention_norm)")
print(f"    │       │")
print(f"    │       └──→ Attention + RoPE")
print(f"    │               │")
print(f"    └───────────(+) ← residual connection")
print(f"                │")
print(f"    ┌───────── h: {list(out.shape)}")
print(f"    │           │")
print(f"    │           ├──→ RMSNorm (ffn_norm)")
print(f"    │           │       │")
print(f"    │           │       └──→ SwiGLU FFN")
print(f"    │           │               │")
print(f"    └───────────────────────(+) ← residual connection")
print(f"                                │")
print(f"                          Output: {list(out.shape)}")

# Count parameters
block_params = sum(p.numel() for p in block.parameters())
attn_params = sum(p.numel() for p in block.attention.parameters())
ffn_params = sum(p.numel() for p in block.feed_forward.parameters())
norm_params = sum(p.numel() for p in block.attention_norm.parameters()) + \
              sum(p.numel() for p in block.ffn_norm.parameters())
print(f"\n--- Parameter Breakdown ---")
print(f"Attention:  {attn_params:>10,} ({100*attn_params/block_params:.1f}%)")
print(f"SwiGLU FFN: {ffn_params:>10,} ({100*ffn_params/block_params:.1f}%)")
print(f"RMSNorms:   {norm_params:>10,} ({100*norm_params/block_params:.1f}%)")
print(f"Total:      {block_params:>10,}")

## 9. Model Configuration — Table 2 (§2.1)

### WHAT
The paper defines four model sizes with specific hyperparameters. We encode these as a Python dataclass and verify that our parameter counts match the paper.

### WHY
Table 2 in the paper specifies the exact configuration for each model size. Matching these numbers validates that our implementation is architecturally correct.

### HOW
From **Table 2** of the paper:

| Model | Params | dim | n_heads | n_layers | Learning Rate |
|-------|--------|-----|---------|----------|--------------|
| LLaMA-7B  | 6.7B  | 4096 | 32 | 32 | 3.0 × 10⁻⁴ |
| LLaMA-13B | 13.0B | 5120 | 40 | 40 | 3.0 × 10⁻⁴ |
| LLaMA-33B | 32.5B | 6656 | 52 | 60 | 1.5 × 10⁻⁴ |
| LLaMA-65B | 65.2B | 8192 | 64 | 80 | 1.5 × 10⁻⁴ |

Additional details:
- **Vocabulary size**: 32,000 (SentencePiece BPE tokenizer)
- **Max sequence length**: 2048 tokens
- **No bias** in any linear layer
- **multiple_of**: 256 (for FFN hidden dim alignment)
- **norm_eps**: 1e-6

### WHERE
Table 2, §2.1.

In [ ]:
# ============================================================
# Model Configuration (Table 2)
# ============================================================

@dataclass
class ModelArgs:
    """
    LLaMA model hyperparameters.
    
    All values correspond to Table 2 of the paper.
    Default values are for LLaMA-7B.
    """
    dim: int = 4096            # Model/embedding dimension
    n_layers: int = 32         # Number of Transformer blocks
    n_heads: int = 32          # Number of attention heads
    vocab_size: int = 32000    # SentencePiece BPE vocabulary
    multiple_of: int = 256     # FFN hidden dim rounded to this
    norm_eps: float = 1e-6     # RMSNorm epsilon
    max_seq_len: int = 2048    # Maximum sequence length
    max_batch_size: int = 32   # Maximum batch size (for KV cache)


def compute_param_count(args: ModelArgs) -> dict:
    """
    Compute the parameter count for a LLaMA model configuration.
    
    Components:
        1. Token embeddings: vocab_size × dim
        2. Per layer:
           a. Attention: 4 × dim² (wq, wk, wv, wo)
           b. FFN: 3 × dim × hidden_dim (w1, w2, w3 for SwiGLU)
           c. 2 × RMSNorm: 2 × dim
        3. Final RMSNorm: dim
        4. Output head: dim × vocab_size
    """
    # Compute SwiGLU hidden dim (matching the FeedForward class)
    hidden_dim = 4 * args.dim
    hidden_dim = int(2 * hidden_dim / 3)
    hidden_dim = args.multiple_of * ((hidden_dim + args.multiple_of - 1) // args.multiple_of)
    
    embedding = args.vocab_size * args.dim
    attn_per_layer = 4 * args.dim * args.dim  # wq + wk + wv + wo
    ffn_per_layer = 3 * args.dim * hidden_dim  # w1 + w2 + w3
    norm_per_layer = 2 * args.dim  # attention_norm + ffn_norm
    final_norm = args.dim
    output_head = args.dim * args.vocab_size  # Often tied with embeddings
    
    total_per_layer = attn_per_layer + ffn_per_layer + norm_per_layer
    total = embedding + args.n_layers * total_per_layer + final_norm + output_head
    
    return {
        'embedding': embedding,
        'attn_per_layer': attn_per_layer,
        'ffn_per_layer': ffn_per_layer,
        'ffn_hidden_dim': hidden_dim,
        'norm_per_layer': norm_per_layer,
        'final_norm': final_norm,
        'output_head': output_head,
        'total_per_layer': total_per_layer,
        'total': total
    }


# ============================================================
# Define all four model sizes from Table 2
# ============================================================
configs = {
    '7B':  ModelArgs(dim=4096, n_layers=32, n_heads=32),
    '13B': ModelArgs(dim=5120, n_layers=40, n_heads=40),
    '33B': ModelArgs(dim=6656, n_layers=60, n_heads=52),
    '65B': ModelArgs(dim=8192, n_layers=80, n_heads=64),
}

paper_params = {  # From Table 2 (in billions)
    '7B': 6.7, '13B': 13.0, '33B': 32.5, '65B': 65.2
}

print("=" * 75)
print("LLaMA Model Configurations (Table 2)")
print("=" * 75)
print(f"{'Model':<8} {'dim':>6} {'heads':>6} {'layers':>7} {'FFN hidden':>11} "
      f"{'Computed':>12} {'Paper':>10} {'Match':>6}")
print("-" * 75)

for name, args in configs.items():
    counts = compute_param_count(args)
    computed_B = counts['total'] / 1e9
    paper_B = paper_params[name]
    match = '✓' if abs(computed_B - paper_B) / paper_B < 0.05 else '✗'
    print(f"{name:<8} {args.dim:>6} {args.n_heads:>6} {args.n_layers:>7} "
          f"{counts['ffn_hidden_dim']:>11,} {computed_B:>11.2f}B {paper_B:>9.1f}B {match:>6}")

# Detailed breakdown for 7B
print(f"\n{'='*60}")
print("Detailed Breakdown: LLaMA-7B")
print(f"{'='*60}")
counts_7b = compute_param_count(configs['7B'])
for k, v in counts_7b.items():
    if k != 'total':
        print(f"  {k:<20}: {v:>15,}")
print(f"  {'TOTAL':<20}: {counts_7b['total']:>15,} ({counts_7b['total']/1e9:.2f}B)")

## 10. The Full LLaMA Model (§2)

### WHAT
The complete LLaMA model stacks all components into an autoregressive Transformer language model:

$$\text{Token IDs} \xrightarrow{\text{Embedding}} \xrightarrow{N \times \text{TransformerBlock}} \xrightarrow{\text{RMSNorm}} \xrightarrow{\text{Linear}} \text{Logits}$$

### WHY
This is the standard decoder-only Transformer architecture (GPT-style), with LLaMA's specific modifications (RMSNorm, SwiGLU, RoPE, no bias). The model is trained as a **language model**: given a sequence of tokens, predict the next token.

### HOW
1. **Token Embedding**: Map integer token IDs to dense vectors of dimension `dim`
2. **$N$ Transformer Blocks**: Each block applies attention + FFN with residual connections
3. **Final RMSNorm**: Normalize the output of the last block
4. **Output Linear Head**: Project from `dim` to `vocab_size` to produce logits

Notable details:
- **No absolute positional embedding**: RoPE in every layer handles position
- **No dropout**: LLaMA does not use dropout anywhere
- **No bias**: All linear layers have `bias=False`

### WHERE
§2 describes the overall architecture. The model follows the decoder-only Transformer pattern from Radford et al. (2018) with the modifications listed in §2.2.

In [ ]:
# ============================================================
# Full LLaMA Model (§2)
# ============================================================

class Transformer(nn.Module):
    """
    The complete LLaMA Transformer language model.
    
    Architecture:
        Token IDs → Embedding → N × TransformerBlock → RMSNorm → Linear → Logits
    
    Key LLaMA design choices:
        - No absolute positional embeddings (RoPE in each layer)
        - Pre-normalization with RMSNorm
        - SwiGLU activation in FFN
        - No dropout anywhere
        - No bias in any linear layer
    """
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        
        # Token embedding table: maps token IDs to vectors
        self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)
        
        # Stack of N Transformer blocks
        self.layers = nn.ModuleList([
            TransformerBlock(
                dim=args.dim,
                n_heads=args.n_heads,
                hidden_dim=4 * args.dim,  # Will be adjusted inside FeedForward
                multiple_of=args.multiple_of,
                norm_eps=args.norm_eps
            )
            for _ in range(args.n_layers)
        ])
        
        # Final RMSNorm (applied after the last Transformer block)
        self.norm = RMSNorm(args.dim, eps=args.norm_eps)
        
        # Output projection: dim → vocab_size (produces logits)
        self.output = nn.Linear(args.dim, args.vocab_size, bias=False)
        
        # Precompute RoPE frequencies for the maximum sequence length
        # head_dim = dim / n_heads
        self.freqs_cis = precompute_freqs_cis(
            args.dim // args.n_heads,  # Head dimension
            args.max_seq_len           # Max positions to precompute
        )
    
    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """
        Forward pass: tokens → logits.
        
        Args:
            tokens: (batch, seq_len) integer token IDs
        
        Returns:
            logits: (batch, seq_len, vocab_size) unnormalized predictions
        """
        batch, seq_len = tokens.shape
        
        # Step 1: Token embedding (no positional embedding!)
        h = self.tok_embeddings(tokens)  # (batch, seq_len, dim)
        
        # Get RoPE frequencies for this sequence length
        freqs_cis = self.freqs_cis[:seq_len].to(h.device)
        
        # Create causal mask
        mask = torch.full((seq_len, seq_len), float("-inf"), device=h.device)
        mask = torch.triu(mask, diagonal=1)
        mask = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)
        
        # Step 2: Pass through all Transformer blocks
        for layer in self.layers:
            h = layer(h, freqs_cis, mask)
        
        # Step 3: Final normalization
        h = self.norm(h)
        
        # Step 4: Project to vocabulary
        logits = self.output(h)  # (batch, seq_len, vocab_size)
        
        return logits


# ============================================================
# Instantiate a SMALL LLaMA for demonstration
# (Full 7B would require ~26GB just for parameters)
# ============================================================
print("=" * 60)
print("Full LLaMA Model (Small Demo Version)")
print("=" * 60)

# Small config for demo purposes
demo_args = ModelArgs(
    dim=256,          # Small embedding dim
    n_layers=4,       # Just 4 layers
    n_heads=4,        # 4 attention heads (head_dim = 64)
    vocab_size=1000,  # Tiny vocab
    max_seq_len=128,  # Short sequences
    multiple_of=64,   # Smaller alignment
)

model = Transformer(demo_args)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nDemo model configuration:")
print(f"  dim = {demo_args.dim}, n_layers = {demo_args.n_layers}, "
      f"n_heads = {demo_args.n_heads}, vocab = {demo_args.vocab_size}")
print(f"  Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")

# Test forward pass
dummy_tokens = torch.randint(0, demo_args.vocab_size, (2, 16))  # (batch=2, seq=16)
logits = model(dummy_tokens)

print(f"\nForward pass:")
print(f"  Input tokens shape:  {dummy_tokens.shape}")
print(f"  Output logits shape: {logits.shape}")
print(f"  Expected:            (2, 16, {demo_args.vocab_size})")

# Show model structure
print(f"\n--- Model Structure ---")
for name, module in model.named_children():
    if isinstance(module, nn.ModuleList):
        print(f"  {name}: {len(module)} × TransformerBlock")
        # Show first block's structure
        for sub_name, sub_module in module[0].named_children():
            params = sum(p.numel() for p in sub_module.parameters())
            print(f"    └─ {sub_name}: {type(sub_module).__name__} ({params:,} params)")
    else:
        params = sum(p.numel() for p in module.parameters())
        print(f"  {name}: {type(module).__name__} ({params:,} params)")

## 11. Cosine Learning Rate Schedule (§2.3)

### WHAT
LLaMA uses a **cosine learning rate schedule** with linear warmup. After warmup, the learning rate decays from $\eta_{\text{max}}$ to $\eta_{\text{min}}$ following a cosine curve.

### WHY
Cosine decay provides a **smooth transition** from high learning rates (for rapid initial learning) to low learning rates (for fine-grained convergence). Compared to step decay:
- No abrupt drops that can destabilize training
- Naturally spends more time at low learning rates near the end

### HOW
The schedule has two phases:

**Phase 1: Linear warmup** (first 2000 steps):
$$\eta_t = \eta_{\text{max}} \cdot \frac{t}{T_{\text{warmup}}}$$

**Phase 2: Cosine decay** (remaining steps):
$$\eta_t = \eta_{\text{min}} + \frac{1}{2}(\eta_{\text{max}} - \eta_{\text{min}})\left(1 + \cos\left(\frac{\pi \cdot (t - T_{\text{warmup}})}{T_{\text{total}} - T_{\text{warmup}}}\right)\right)$$

The final learning rate is set to $\eta_{\text{min}} = 0.1 \times \eta_{\text{max}}$ (10% of peak).

**Optimizer**: AdamW with:
- $\beta_1 = 0.9$, $\beta_2 = 0.95$
- Weight decay: $0.1$
- Gradient clipping: $1.0$

### WHERE
§2.3: *"We use a cosine learning rate schedule, such that the final learning rate is equal to 10% of the maximal learning rate. We use a weight decay of 0.1 and gradient clipping of 1.0."*

In [ ]:
# ============================================================
# Cosine Learning Rate Schedule with Warmup (§2.3)
# ============================================================

def cosine_lr_schedule(
    step: int,
    max_lr: float,
    min_lr: float,
    warmup_steps: int,
    total_steps: int
) -> float:
    """
    Compute learning rate at a given step.
    
    Phase 1 (step < warmup_steps): Linear warmup
        η = max_lr * step / warmup_steps
    
    Phase 2 (step >= warmup_steps): Cosine decay
        η = min_lr + 0.5 * (max_lr - min_lr) * (1 + cos(π * progress))
        where progress = (step - warmup) / (total - warmup)
    """
    if step < warmup_steps:
        # Linear warmup: 0 → max_lr over warmup_steps
        return max_lr * step / warmup_steps
    elif step >= total_steps:
        # Past the end: use minimum LR
        return min_lr
    else:
        # Cosine decay from max_lr to min_lr
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))


# ============================================================
# Visualization: LR schedule for different model sizes
# ============================================================

# Training details from the paper
# LLaMA-7B: 1T tokens, batch size 4M tokens, ~250K steps
# LLaMA-65B: 1.4T tokens, batch size 4M tokens, ~350K steps
warmup_steps = 2000
total_steps_7b = 250_000   # ~1T tokens / 4M batch
total_steps_65b = 350_000  # ~1.4T tokens / 4M batch

# LR values from Table 2
lr_configs = {
    '7B/13B':  {'max_lr': 3e-4, 'total_steps': total_steps_7b, 'color': '#3498db'},
    '33B/65B': {'max_lr': 1.5e-4, 'total_steps': total_steps_65b, 'color': '#e74c3c'},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Full schedule
for name, cfg in lr_configs.items():
    steps = np.arange(0, cfg['total_steps'])
    min_lr = cfg['max_lr'] * 0.1  # Final LR = 10% of max
    lrs = [cosine_lr_schedule(s, cfg['max_lr'], min_lr, warmup_steps, cfg['total_steps'])
           for s in steps]
    axes[0].plot(steps / 1000, lrs, label=f"LLaMA-{name}", color=cfg['color'], linewidth=2)

axes[0].set_xlabel('Training Steps (×1000)', fontsize=12)
axes[0].set_ylabel('Learning Rate', fontsize=12)
axes[0].set_title('Cosine LR Schedule with Warmup (§2.3)', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Right: Zoomed into warmup
for name, cfg in lr_configs.items():
    steps = np.arange(0, 5000)
    min_lr = cfg['max_lr'] * 0.1
    lrs = [cosine_lr_schedule(s, cfg['max_lr'], min_lr, warmup_steps, cfg['total_steps'])
           for s in steps]
    axes[1].plot(steps, lrs, label=f"LLaMA-{name}", color=cfg['color'], linewidth=2)

axes[1].axvline(x=warmup_steps, color='gray', linestyle='--', label='End of warmup', alpha=0.7)
axes[1].set_xlabel('Training Steps', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('Warmup Phase (First 5000 Steps)', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.show()

print("Training Hyperparameters (§2.3):")
print("  Optimizer: AdamW")
print("  β₁ = 0.9, β₂ = 0.95")
print("  Weight decay: 0.1")
print("  Gradient clipping: 1.0")
print(f"  Warmup steps: {warmup_steps:,}")
print(f"  Final LR: 10% of peak LR")

## 12. Training Data Mix — Table 1 (§2.1)

### WHAT
LLaMA is trained on a mixture of **publicly available datasets** totaling approximately **1.4 trillion tokens**. A key contribution of the paper is demonstrating that state-of-the-art performance is achievable **without proprietary data**.

### WHY
From §1: *"Unlike Chinchilla, PaLM, or GPT-3, we only use publicly available data, making our work compatible with open-sourcing."*

The data mixture is carefully designed:
- **CommonCrawl** (67%): Massive web text, filtered with a CCNet pipeline + classifier
- **C4** (15%): The Colossal Clean Crawled Corpus, already heavily filtered
- **GitHub** (4.5%): Code data for code generation capabilities
- **Wikipedia** (4.5%): High-quality encyclopedic knowledge (20 languages)
- **Books** (4.5%): Gutenberg + Books3 for long-form text understanding
- **ArXiv** (2.5%): Scientific papers for math and reasoning
- **StackExchange** (2%): Q&A data for instruction following

### HOW
Different sources are sampled at different rates. Some sources are seen **more than once** (multiple epochs) while others are seen less than once:

| Dataset | Sampling % | Epochs (1.4T tokens) | Disk Size |
|---------|-----------|---------------------|----------|
| CommonCrawl | 67.0% | 1.10 | 3.3 TB |
| C4 | 15.0% | 1.06 | 783 GB |
| GitHub | 4.5% | 0.64 | 328 GB |
| Wikipedia | 4.5% | 2.45 | 83 GB |
| Books | 4.5% | 2.23 | 85 GB |
| ArXiv | 2.5% | 1.06 | 92 GB |
| StackExchange | 2.0% | 1.03 | 78 GB |

### WHERE
Table 1, §2.1 (Pre-training Data).

In [ ]:
# ============================================================
# Training Data Mix Visualization (Table 1, §2.1)
# ============================================================

# Data from Table 1 of the paper
data_sources = {
    'CommonCrawl':  {'pct': 67.0, 'epochs': 1.10, 'size_gb': 3300, 'tokens_b': 927},
    'C4':           {'pct': 15.0, 'epochs': 1.06, 'size_gb': 783,  'tokens_b': 198},
    'GitHub':       {'pct': 4.5,  'epochs': 0.64, 'size_gb': 328,  'tokens_b': 100},
    'Wikipedia':    {'pct': 4.5,  'epochs': 2.45, 'size_gb': 83,   'tokens_b': 25},
    'Books':        {'pct': 4.5,  'epochs': 2.23, 'size_gb': 85,   'tokens_b': 27},
    'ArXiv':        {'pct': 2.5,  'epochs': 1.06, 'size_gb': 92,   'tokens_b': 33},
    'StackExchange': {'pct': 2.0, 'epochs': 1.03, 'size_gb': 78,   'tokens_b': 27},
}

names = list(data_sources.keys())
pcts = [d['pct'] for d in data_sources.values()]
epochs = [d['epochs'] for d in data_sources.values()]
sizes = [d['size_gb'] for d in data_sources.values()]
tokens = [d['tokens_b'] for d in data_sources.values()]

# Color palette
colors = ['#3498db', '#2ecc71', '#9b59b6', '#f39c12', '#e74c3c', '#1abc9c', '#e67e22']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Left: Pie chart of sampling proportions
wedges, texts, autotexts = axes[0].pie(
    pcts, labels=names, autopct='%1.1f%%',
    colors=colors, startangle=90, pctdistance=0.85,
    textprops={'fontsize': 9}
)
for autotext in autotexts:
    autotext.set_fontsize(8)
axes[0].set_title('Sampling Proportions (Table 1)', fontsize=13)

# Middle: Bar chart of epochs per source
bars = axes[1].barh(names[::-1], epochs[::-1], color=colors[::-1], edgecolor='white')
axes[1].axvline(x=1.0, color='red', linestyle='--', alpha=0.7, label='1 epoch')
axes[1].set_xlabel('Epochs (for 1.4T token budget)', fontsize=12)
axes[1].set_title('Epochs per Source', fontsize=13)
axes[1].legend(fontsize=10)
# Add value labels
for bar, val in zip(bars, epochs[::-1]):
    axes[1].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=10)

# Right: Tokens per source
bars2 = axes[2].barh(names[::-1], tokens[::-1], color=colors[::-1], edgecolor='white')
axes[2].set_xlabel('Tokens (Billions)', fontsize=12)
axes[2].set_title('Tokens per Source', fontsize=13)
for bar, val in zip(bars2, tokens[::-1]):
    axes[2].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                f'{val}B', va='center', fontsize=10)

plt.tight_layout()
plt.show()

# Summary statistics
total_tokens = sum(tokens)
total_size = sum(sizes)
print(f"\nTotal unique tokens: ~{total_tokens}B ({total_tokens/1000:.1f}T)")
print(f"Total disk size: ~{total_size/1000:.1f} TB")
print(f"\nKey insight: Wikipedia sees 2.45 epochs (repeated for quality),")
print(f"while GitHub sees only 0.64 epochs (subsampled — huge corpus, lower quality density)")

## 13. Efficient Implementation Details (§2.4)

### WHAT
The paper describes several engineering optimizations that make training the large LLaMA models feasible on a cluster of 2048 A100-80GB GPUs.

### WHY
Training a 65B parameter model on 1.4T tokens is an enormous computational undertaking. Without careful engineering, memory and communication overhead would make this infeasible.

### HOW
The paper mentions three key optimizations:

1. **Memory-efficient attention** (xformers library):
   - Does NOT store the full $N \times N$ attention matrix
   - Uses FlashAttention-style tiled computation
   - Reduces memory from $O(N^2)$ to $O(N)$

2. **Activation checkpointing** (gradient checkpointing):
   - Instead of storing all intermediate activations for the backward pass, re-compute them during backprop
   - Trades **compute** for **memory**: ~33% more FLOPs but drastically less memory
   - Specifically: save only the input to each Transformer block, recompute everything inside during backward

3. **Model & Sequence Parallelism**:
   - Tensor parallelism across GPUs within a node
   - Sequence parallelism for the normalization layers
   - Reduces memory per GPU and communication volume

### WHERE
§2.4: *"We make several optimizations to improve the training speed of our models."*

### Training Speed
- **380 tokens/sec/GPU** on A100-80GB
- **2048 A100 GPUs** for the 65B model
- **~21 days** for 1.4T tokens on the 65B model
- Total compute: **1,022,362 GPU-hours** for the 65B model

In [ ]:
# ============================================================
# Activation Checkpointing: Memory Savings Demo (§2.4)
# ============================================================
# This demonstrates the core idea behind gradient checkpointing:
# Instead of storing all intermediate activations, recompute them.

def estimate_memory_no_checkpoint(dim: int, n_layers: int, seq_len: int, batch: int) -> dict:
    """
    Estimate activation memory WITHOUT checkpointing.
    
    Each layer stores:
    - Input to attention norm: batch × seq × dim
    - Q, K, V: 3 × batch × seq × dim
    - Attention scores: batch × n_heads × seq × seq (the big one!)
    - Attention output: batch × seq × dim
    - FFN intermediates: batch × seq × ffn_dim
    - Various other intermediates
    
    Simplified estimate: ~12 × batch × seq × dim per layer (for float16)
    Plus the attention matrix: batch × n_heads × seq² (per layer)
    """
    bytes_per_element = 2  # float16
    n_heads = dim // 64    # Approximate
    
    # Per-layer activation memory (simplified)
    activation_per_layer = (
        12 * batch * seq_len * dim +          # Linear intermediates
        batch * n_heads * seq_len * seq_len    # Attention matrix (O(n²)!)
    ) * bytes_per_element
    
    total = activation_per_layer * n_layers
    return {
        'per_layer_mb': activation_per_layer / 1e6,
        'total_mb': total / 1e6,
        'total_gb': total / 1e9
    }


def estimate_memory_with_checkpoint(dim: int, n_layers: int, seq_len: int, batch: int) -> dict:
    """
    Estimate activation memory WITH checkpointing.
    
    Only stores the INPUT to each layer (not intermediates).
    Everything else is recomputed during the backward pass.
    
    Stored: n_layers × batch × seq × dim
    Plus ONE layer's worth of intermediates (for the current backward computation)
    """
    bytes_per_element = 2
    n_heads = dim // 64
    
    # Only store layer inputs
    checkpoint_stored = n_layers * batch * seq_len * dim * bytes_per_element
    
    # Plus one layer's intermediates (recomputed during backward)
    one_layer = (
        12 * batch * seq_len * dim +
        batch * n_heads * seq_len * seq_len
    ) * bytes_per_element
    
    total = checkpoint_stored + one_layer
    return {
        'checkpoint_mb': checkpoint_stored / 1e6,
        'one_layer_mb': one_layer / 1e6,
        'total_mb': total / 1e6,
        'total_gb': total / 1e9
    }


# Compare for LLaMA-7B parameters
dim = 4096
n_layers = 32
seq_len = 2048
batch = 1  # Per-GPU batch

no_ckpt = estimate_memory_no_checkpoint(dim, n_layers, seq_len, batch)
with_ckpt = estimate_memory_with_checkpoint(dim, n_layers, seq_len, batch)

print("=" * 60)
print("Activation Memory: Checkpointing vs No Checkpointing")
print(f"(LLaMA-7B: dim={dim}, layers={n_layers}, seq={seq_len})")
print("=" * 60)

print(f"\n--- Without Checkpointing ---")
print(f"  Per layer: {no_ckpt['per_layer_mb']:.1f} MB")
print(f"  Total:     {no_ckpt['total_gb']:.1f} GB")

print(f"\n--- With Checkpointing ---")
print(f"  Stored checkpoints: {with_ckpt['checkpoint_mb']:.1f} MB")
print(f"  One layer recompute: {with_ckpt['one_layer_mb']:.1f} MB")
print(f"  Total:     {with_ckpt['total_gb']:.1f} GB")

savings = (1 - with_ckpt['total_gb'] / no_ckpt['total_gb']) * 100
print(f"\n  Memory savings: {savings:.1f}%")
print(f"  Compute overhead: ~33% more FLOPs (recompute forward in backward pass)")

# Training speed stats from the paper
print(f"\n{'='*60}")
print("Training Speed Statistics (§2.4)")
print(f"{'='*60}")
print(f"  Throughput: 380 tokens/sec/GPU (A100-80GB)")
print(f"  LLaMA-65B: 2048 A100 GPUs")
print(f"  LLaMA-65B: 1,022,362 GPU-hours for 1.4T tokens")
print(f"  LLaMA-65B: ~21 days wall-clock time")
print(f"  LLaMA-7B:  82,432 GPU-hours for 1T tokens")

## 14. Benchmark Results (§3)

### WHAT
The paper evaluates LLaMA on a comprehensive suite of **20 benchmarks** spanning common sense reasoning, reading comprehension, code generation, math, and massive multitask evaluation (MMLU).

### WHY
The central claim is that LLaMA models, despite being smaller, match or exceed much larger models:
- **LLaMA-13B outperforms GPT-3 (175B)** on most benchmarks
- **LLaMA-65B is competitive with Chinchilla-70B and PaLM-540B**

This validates the core thesis: training smaller models on more data is more efficient at inference time.

### HOW
Key evaluation results from the paper:

**Common Sense Reasoning** (Table 3 — 0-shot):
| Model | BoolQ | PIQA | SIQA | HellaSwag | WinoGrande | ARC-e | ARC-c | OBQA |
|-------|-------|------|------|-----------|------------|-------|-------|------|
| GPT-3 175B | 60.5 | 81.0 | — | 78.9 | 70.2 | 68.8 | 51.4 | 57.6 |
| LLaMA-7B | 76.5 | 79.8 | 48.9 | 76.1 | 70.1 | 72.8 | 47.6 | 57.2 |
| LLaMA-13B | 78.1 | 80.1 | 50.4 | 79.2 | 73.0 | 74.8 | 52.7 | 56.4 |
| LLaMA-65B | 85.3 | 82.8 | 52.3 | 84.2 | 77.0 | 78.9 | 56.0 | 60.2 |

### WHERE
§3 (Main Results), Tables 3-9.

In [ ]:
# ============================================================
# Benchmark Results Visualization (§3, Tables 3-9)
# ============================================================

# Key results from the paper (selected benchmarks)
benchmarks = {
    'Common Sense (avg)': {
        'GPT-3 175B': 66.9, 'Chinchilla 70B': 78.7, 'PaLM 540B': 80.5,
        'LLaMA-7B': 66.1, 'LLaMA-13B': 68.1, 'LLaMA-33B': 72.7, 'LLaMA-65B': 72.1
    },
    'HellaSwag': {
        'GPT-3 175B': 78.9, 'Chinchilla 70B': 80.8, 'PaLM 540B': 83.4,
        'LLaMA-7B': 76.1, 'LLaMA-13B': 79.2, 'LLaMA-33B': 82.8, 'LLaMA-65B': 84.2
    },
    'MMLU (5-shot)': {
        'GPT-3 175B': 43.9, 'Chinchilla 70B': 67.6, 'PaLM 540B': 69.3,
        'LLaMA-7B': 35.1, 'LLaMA-13B': 46.9, 'LLaMA-33B': 57.8, 'LLaMA-65B': 63.4
    },
    'TriviaQA': {
        'GPT-3 175B': 64.3, 'Chinchilla 70B': 72.3, 'PaLM 540B': 81.4,
        'LLaMA-7B': 63.6, 'LLaMA-13B': 73.0, 'LLaMA-33B': 78.1, 'LLaMA-65B': 81.7
    },
    'HumanEval\n(pass@1)': {
        'GPT-3 175B': 0.0, 'Chinchilla 70B': 16.0, 'PaLM 540B': 26.2,
        'LLaMA-7B': 10.5, 'LLaMA-13B': 15.8, 'LLaMA-33B': 21.7, 'LLaMA-65B': 23.7
    },
}

# Grouped bar chart
fig, ax = plt.subplots(figsize=(16, 7))

model_names = ['GPT-3 175B', 'Chinchilla 70B', 'PaLM 540B',
               'LLaMA-7B', 'LLaMA-13B', 'LLaMA-33B', 'LLaMA-65B']
model_colors = ['#95a5a6', '#7f8c8d', '#bdc3c7',  # Gray tones for baselines
                '#3498db', '#2980b9', '#1f6dad', '#19537d']  # Blue gradient for LLaMA

bench_names = list(benchmarks.keys())
n_models = len(model_names)
n_benchmarks = len(bench_names)
x = np.arange(n_benchmarks)
width = 0.11  # Width of each bar

for i, (model, color) in enumerate(zip(model_names, model_colors)):
    values = [benchmarks[b].get(model, 0) for b in bench_names]
    offset = (i - n_models / 2 + 0.5) * width
    bars = ax.bar(x + offset, values, width, label=model, color=color,
                  edgecolor='white', linewidth=0.5)

ax.set_xlabel('Benchmark', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('LLaMA vs Baselines: Key Benchmark Results (§3)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(bench_names, fontsize=10)
ax.legend(fontsize=9, ncol=4, loc='upper left')
ax.grid(True, alpha=0.2, axis='y')
ax.set_ylim(0, 95)

plt.tight_layout()
plt.show()

# Highlight key comparisons
print("\n" + "=" * 60)
print("Key Findings (§3)")
print("=" * 60)
print("")
print("1. LLaMA-13B (13B params) outperforms GPT-3 (175B params) on")
print("   most benchmarks — with 13× fewer parameters!")
print("")
print("2. LLaMA-65B (65B params) is competitive with Chinchilla (70B)")
print("   and approaches PaLM (540B) on many tasks.")
print("")
print("3. MMLU is where LLaMA lags most — likely because the training")
print("   data lacks enough MMLU-style Q&A format data.")
print("")
print("4. On TriviaQA, LLaMA-65B (81.7) actually beats PaLM-540B (81.4)!")

## 15. Training Loss Curves (Approximate Figure 1)

### WHAT
Figure 1 of the paper shows the **training loss** (cross-entropy on the next-token prediction task) decreasing over the course of training for all four model sizes.

### WHY
The training curves demonstrate two key points:
1. **Larger models converge to lower loss** — the scaling law relationship holds
2. **Loss is still decreasing at the end** — the models have not fully converged, validating LLaMA's thesis that training on more data would continue to improve performance

### HOW
We generate **synthetic curves** that match the general shape and final values from Figure 1. The real curves show:
- Initial rapid decrease (steep learning)
- Gradual convergence (diminishing returns)
- Final values: ~1.70 (7B), ~1.60 (13B), ~1.48 (33B), ~1.40 (65B)

### WHERE
Figure 1, §2.3.

In [ ]:
# ============================================================
# Training Loss Curves (Approximating Figure 1)
# ============================================================

def synthetic_loss_curve(total_tokens_T: float, final_loss: float,
                         initial_loss: float = 10.0, n_points: int = 500) -> tuple:
    """
    Generate a synthetic training loss curve that matches the general
    shape of LLaMA's Figure 1.
    
    The curve follows: L(t) = final + (initial - final) * exp(-k * t^0.5)
    with small random noise for realism.
    """
    tokens = np.linspace(0.001, total_tokens_T, n_points)  # Tokens in trillions
    
    # The exponent k controls how fast the curve decays
    # Calibrate so that the final point reaches approximately final_loss
    k = -np.log((final_loss - final_loss * 0.02) / (initial_loss - final_loss)) / (total_tokens_T ** 0.5)
    
    # Base curve: exponential decay with sqrt-time
    loss = final_loss + (initial_loss - final_loss) * np.exp(-k * tokens ** 0.5)
    
    # Add small noise for realism
    noise = np.random.normal(0, 0.01, n_points)
    loss = loss + noise
    
    return tokens, loss


# Model configurations matching Figure 1
models_fig1 = {
    'LLaMA-7B':  {'tokens_T': 1.0,  'final_loss': 1.70, 'color': '#3498db'},
    'LLaMA-13B': {'tokens_T': 1.0,  'final_loss': 1.60, 'color': '#2ecc71'},
    'LLaMA-33B': {'tokens_T': 1.4,  'final_loss': 1.48, 'color': '#e74c3c'},
    'LLaMA-65B': {'tokens_T': 1.4,  'final_loss': 1.40, 'color': '#9b59b6'},
}

fig, ax = plt.subplots(figsize=(12, 6))

for name, cfg in models_fig1.items():
    tokens, loss = synthetic_loss_curve(cfg['tokens_T'], cfg['final_loss'])
    ax.plot(tokens, loss, label=name, color=cfg['color'], linewidth=2, alpha=0.9)

ax.set_xlabel('Tokens (Trillions)', fontsize=13)
ax.set_ylabel('Training Loss (Cross-Entropy)', fontsize=13)
ax.set_title('Training Loss Curves (Approximating Figure 1)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1.5)
ax.set_ylim(1.2, 3.5)

# Add annotations
ax.annotate('7B & 13B trained on 1T tokens', xy=(1.0, 1.65), fontsize=10,
            ha='center', color='gray', style='italic')
ax.annotate('33B & 65B trained on 1.4T tokens', xy=(1.4, 1.44), fontsize=10,
            ha='right', color='gray', style='italic')

plt.tight_layout()
plt.show()

print("Key observations from Figure 1:")
print("• All curves are still decreasing — more tokens would continue to help")
print("• Larger models achieve lower loss throughout training")
print("• The gap between model sizes narrows with more training tokens")
print("• This validates the thesis: smaller models benefit greatly from more data")

## 16. Carbon Footprint (§6)

### WHAT
The paper transparently reports the **energy consumption and carbon footprint** of training all LLaMA models, following responsible AI practices.

### WHY
Large language model training has significant environmental impact. By reporting these numbers, the authors enable informed discussion about the cost-benefit tradeoff of large-scale training. Importantly, they argue that training one large open model (which can be reused by many researchers) is **more efficient** than many groups independently training their own.

### HOW
Energy consumption is estimated from GPU-hours:

$$\text{Energy (Wh)} = \text{GPU-hours} \times \text{TDP per GPU}$$

Carbon emissions depend on the **Power Usage Effectiveness (PUE)** of the data center and the **carbon intensity** of the electrical grid:

$$\text{CO}_2\text{eq (tons)} = \frac{\text{Energy (MWh)} \times \text{PUE} \times \text{Carbon Intensity (kg/MWh)}}{1000}$$

### WHERE
§6 (Carbon Footprint), Table 12.

In [ ]:
# ============================================================
# Carbon Footprint Analysis (§6, Table 12)
# ============================================================

# Data from Table 12 of the paper
carbon_data = {
    'LLaMA-7B':  {'gpu_hours': 82_432,    'gpu_type': 'A100-80GB', 'tdp_w': 400, 'n_gpus': 256},
    'LLaMA-13B': {'gpu_hours': 135_168,   'gpu_type': 'A100-80GB', 'tdp_w': 400, 'n_gpus': 512},
    'LLaMA-33B': {'gpu_hours': 530_432,   'gpu_type': 'A100-80GB', 'tdp_w': 400, 'n_gpus': 1024},
    'LLaMA-65B': {'gpu_hours': 1_022_362, 'gpu_type': 'A100-80GB', 'tdp_w': 400, 'n_gpus': 2048},
}

# Meta's data center PUE and US carbon intensity (from the paper)
PUE = 1.1           # Power Usage Effectiveness
CARBON_INTENSITY = 0.385  # kg CO2 per kWh (US average grid)

print("=" * 70)
print("Carbon Footprint of LLaMA Training (§6, Table 12)")
print("=" * 70)
print(f"\n{'Model':<12} {'GPU-hrs':>12} {'Energy(MWh)':>12} {'CO₂eq(tons)':>13} {'Wall days':>10}")
print("-" * 70)

total_gpu_hours = 0
total_energy = 0
total_co2 = 0

for name, data in carbon_data.items():
    # Energy = GPU-hours × TDP (in Wh) → convert to MWh
    energy_mwh = data['gpu_hours'] * data['tdp_w'] / 1e6
    
    # CO2 = Energy × PUE × Carbon Intensity
    co2_tons = energy_mwh * PUE * CARBON_INTENSITY / 1  # Already in correct units
    
    # Wall-clock time in days
    wall_days = data['gpu_hours'] / data['n_gpus'] / 24
    
    total_gpu_hours += data['gpu_hours']
    total_energy += energy_mwh
    total_co2 += co2_tons
    
    print(f"{name:<12} {data['gpu_hours']:>12,} {energy_mwh:>12.1f} {co2_tons:>13.1f} {wall_days:>10.1f}")

print("-" * 70)
print(f"{'TOTAL':<12} {total_gpu_hours:>12,} {total_energy:>12.1f} {total_co2:>13.1f}")

# Context and comparisons
print(f"\n{'='*70}")
print("Context & Comparisons")
print(f"{'='*70}")
print(f"\nLLaMA-65B specifically:")
print(f"  GPU-hours:     1,022,362")
print(f"  Energy:        449 MWh (paper's reported value)")
print(f"  Carbon:        173 tCO₂eq (paper's reported value)")
print(f"")
print(f"For comparison:")
print(f"  • GPT-3 (175B):  ~1,287 MWh (estimated)")
print(f"  • PaLM (540B):   ~3,400 MWh (estimated)")
print(f"  • Average US home: ~10.5 MWh/year")
print(f"  • LLaMA-65B uses the equivalent of ~43 US homes' annual electricity")
print(f"")
print(f"Key argument from §6:")
print(f"  'Training open models that can be widely reused is more carbon-efficient")
print(f"   than each research group training their own models from scratch.'")

## 17. Key Takeaways & Impact

### The Five Key Findings of LLaMA

---

**1. Smaller models, more data — the inference-optimal scaling insight**

The Chinchilla scaling laws optimize for *training compute*. But LLaMA shows that if you care about *inference efficiency*, you should train smaller models for longer. A 7B model trained on 1T tokens beats models 10-25× larger trained on fewer tokens.

---

**2. Public data is sufficient for state-of-the-art performance**

Unlike GPT-3, Chinchilla, and PaLM, LLaMA uses **only publicly available data**. This was a critical contribution for reproducibility and open science.

---

**3. Architectural simplicity with targeted improvements**

LLaMA's architecture is a standard Transformer with just three modifications:
- **RMSNorm** (simpler normalization)
- **SwiGLU** (better activation function)
- **RoPE** (relative position encoding)

No novel attention mechanisms, no mixture-of-experts, no retrieval augmentation — just careful engineering of known best practices.

---

**4. LLaMA-13B outperforms GPT-3 (175B) on most benchmarks**

This is perhaps the most striking result: a model with **13× fewer parameters** outperforms one of the most famous language models ever trained. This makes deployment practical on a single GPU.

---

**5. Impact: The foundation of the open LLM ecosystem**

LLaMA catalyzed an explosion of open-source LLM development:
- **Stanford Alpaca** (instruction-tuned LLaMA-7B, $600 to train)
- **Vicuna** (LLaMA fine-tuned on ShareGPT conversations)
- **LLaMA 2** (official successor with RLHF)
- **Code Llama** (specialized for code generation)
- **LLaMA 3** (expanded to 8B/70B/405B with 15T tokens)

LLaMA proved that open models could be competitive with closed-source ones, democratizing LLM research and development.

---

### Architecture Summary

| Component | Original Transformer | LLaMA |
|-----------|---------------------|-------|
| Normalization | LayerNorm (post) | RMSNorm (pre) |
| Position encoding | Sinusoidal / Learned | RoPE (every layer) |
| FFN activation | ReLU | SwiGLU |
| FFN hidden dim | 4d (2 matrices) | 2/3·4d (3 matrices) |
| Bias terms | Yes | No |
| Dropout | Yes | No |
| Attention | Standard MHA | Standard MHA + RoPE |